In [ ]:
# Sesión 3 del módulo 9, clase 04/09
# ============================
#  Instalación y SparkSession
# ============================
!pip -q install pyspark

from pyspark.sql import SparkSession

# Crea/obtiene la sesión de Spark
spark = SparkSession.builder.appName("ClasesRDD").getOrCreate()

# Accede al SparkContext (punto de entrada a RDDs)
sc = spark.sparkContext

print(spark.version, "-> Spark listo")

3.5.1 -> Spark listo


In [ ]:
# Actividad 1: Análisis de transacciones (RDDs + Pair RDD + acciones básicas)
# Objetivos: leer CSV como RDD, limpiar encabezado, mapear a tuplas, filter, reduceByKey, sortBy, y estadísticas simples.
# ============================================
# 1) Crear un CSV de ejemplo (transacciones)
# ============================================
transacciones = """id,usuario,monto,fecha,categoria
1,Ana,1200,2025-03-10,Supermercado
2,Juan,800,2025-03-10,Restaurante
3,Lucia,2200,2025-03-11,Tecnologia
4,Ana,400,2025-03-11,Restaurante
5,Juan,3100,2025-03-12,Tecnologia
6,Lucia,950,2025-03-12,Supermercado
7,Carlos,1800,2025-03-13,Supermercado
8,Carlos,700,2025-03-13,Restaurante
9,Ana,2600,2025-03-14,Tecnologia
"""
# Aquí se crea una cadena multilínea (triple comillas """...""") con contenido en formato CSV.
# Cada fila tiene 5 columnas:
# id (identificador de la transacción)
# usuario (nombre de la persona que hizo la compra)
# monto (dinero gastado en esa transacción)
# fecha (día de la transacción)
# categoría (tipo de gasto, como supermercado o tecnología)

# Guardamos el CSV en /content (entorno Colab)
with open("/content/transacciones_clase.csv", "w", encoding="utf-8") as f:
    f.write(transacciones)
# with open(...) as f: → abre un archivo en una ruta (/content/...) que en Google Colab corresponde al almacenamiento temporal.

# El modo "w" significa write (escritura). Si el archivo existe, lo sobrescribe.
# encoding="utf-8" asegura compatibilidad con acentos o caracteres especiales.
# f.write(transacciones) escribe el contenido de la variable transacciones en el archivo.
# Cuando se cierra el bloque with, el archivo queda grabado como /content/transacciones_clase.csv.

In [ ]:
# ============================================
# 2) Cargar como RDD de líneas (lazy)
#    (las transformaciones no se ejecutan hasta una acción)
# ============================================
rdd = sc.textFile("/content/transacciones_clase.csv")
# sc es el SparkContext, el punto de entrada a RDDs en Spark.
# textFile(...) lee un archivo de texto línea por línea y devuelve un RDD de strings.

# Cada línea del archivo se convierte en un elemento del RDD.

# En este caso, el RDD tendrá 10 elementos: una línea de encabezado y 9 líneas de transacciones.

# Ejemplo de lo que contendría rdd.take(3):
# [
# "id,usuario,monto,fecha,categoria",
# "1,Ana,1200,2025-03-10,Supermercado",
# "2,Juan,800,2025-03-10,Restaurante"
# ]

# Primera línea es encabezado -> la excluimos
header = rdd.first()
# first() es una acción que devuelve el primer elemento del RDD.
# Como el archivo tiene un encabezado (id,usuario,monto,fecha,categoria), esa será la primera línea.
# Ahora, la variable header contiene:
# "id,usuario,monto,fecha,categoria"

# Eliminar el encabezado
rdd_data = rdd.filter(lambda l: l != header)
# filter(...) es una transformación: aplica una función booleana a cada elemento del RDD.
# Aquí usamos una función lambda que mantiene todas las líneas distintas del encabezado.
# lambda l: l != header significa: “incluye esta línea l solo si no es igual al encabezado”.

# Ahora, rdd_data contiene únicamente las filas con transacciones (sin la primera línea de títulos).

In [ ]:
# ============================================
# 3) Transformar: string CSV -> tupla tipada
#    (usuario, monto(float), categoria)
# ============================================
def parse_row(linea):
    # id,usuario,monto,fecha,categoria
    campos = linea.split(",")
# linea.split(",") → divide el string en una lista usando la coma como separador.
# Ejemplo:
# "1,Ana,1200,2025-03-10,Supermercado".split(",")
# → ["1", "Ana", "1200", "2025-03-10", "Supermercado"]

# usuario = campos[1] → toma el segundo campo ("Ana", "Juan", "Lucia"...).
# monto = float(campos[2]) → convierte el tercer campo a número decimal (1200.0, 800.0, etc.).
# categoria = campos[4] → toma el quinto campo ("Supermercado", "Restaurante", "Tecnologia").
# La función devuelve una tupla tipada:
# (usuario, monto, categoria)
# Ejemplo:
# ("Ana", 1200.0, "Supermercado")

    usuario = campos[1]
    monto = float(campos[2])
    categoria = campos[4]
    return (usuario, monto, categoria)

rdd_tx = rdd_data.map(parse_row)
# Aplicar transformación con map
# map(...) aplica la función parse_row a cada línea del RDD.
# Resultado: un nuevo RDD (rdd_tx) que ya no tiene strings crudos, sino tuplas limpias y tipadas.
# Antes (en rdd_data):
# "1,Ana,1200,2025-03-10,Supermercado"
# Después (en rdd_tx):
# ("Ana", 1200.0, "Supermercado")

# Ver 5 elementos (acción: take)
print("Muestra:", rdd_tx.take(5))
# take(5) es una acción: devuelve los primeros 5 elementos del RDD.
# Aquí Spark realmente ejecuta el plan de trabajo: lee el archivo, aplica parse_row, y trae 5 resultados.

# Ejemplo de salida:
# Muestra: [
# ("Ana", 1200.0, "Supermercado"),
# ("Juan", 800.0, "Restaurante"),
# ("Lucia", 2200.0, "Tecnologia"),
# ("Ana", 400.0, "Restaurante"),
# ("Juan", 3100.0, "Tecnologia")
# ]

Muestra: [('Ana', 1200.0, 'Supermercado'), ('Juan', 800.0, 'Restaurante'), ('Lucia', 2200.0, 'Tecnologia'), ('Ana', 400.0, 'Restaurante'), ('Juan', 3100.0, 'Tecnologia')]


('Ana', 1200.0, 'Supermercado') → Ana gastó 1200.0 en Supermercado.

('Juan', 800.0, 'Restaurante') → Juan gastó 800.0 en Restaurante.

('Lucia', 2200.0, 'Tecnologia') → Lucia gastó 2200.0 en Tecnologia.

('Ana', 400.0, 'Restaurante') → Ana gastó 400.0 en Restaurante.

('Juan', 3100.0, 'Tecnologia') → Juan gastó 3100.0 en Tecnologia.

La salida significa que Spark ya convirtió las líneas de texto crudo del CSV en tuplas tipadas, donde:

El primer valor es el usuario.

El segundo valor es el monto (float).

El tercer valor es la categoría de gasto.

In [ ]:
# ============================================
# 4) Filtrar transacciones > 1000
# ============================================
# Hasta ahora, rdd_tx contiene tuplas en la forma:
# (usuario, monto, categoria)
# Ejemplo real:
# ("Ana", 1200.0, "Supermercado")
# ("Juan", 800.0, "Restaurante")
# ("Lucia", 2200.0, "Tecnologia")
rdd_altas = rdd_tx.filter(lambda t: t[1] > 1000.0)
# .filter(...) es una transformación que mantiene solo los elementos que cumplen cierta condición.
# Aquí la condición es lambda t: t[1] > 1000.0, que significa:
# t representa cada tupla (usuario, monto, categoria).
# t[1] corresponde al campo monto.
# Se seleccionan únicamente las transacciones con monto mayor a 1000.
# Ejemplo de lo que ocurre:
# ("Ana", 1200.0, "Supermercado") ✅ (pasa el filtro, porque 1200 > 1000)
# ("Juan", 800.0, "Restaurante") ❌ (se descarta, porque 800 ≤ 1000)
# ("Lucia", 2200.0, "Tecnologia") ✅ (pasa el filtro)

print("Transacciones > 1000:", rdd_altas.collect())
# .collect() es una acción: obliga a Spark a ejecutar el plan de trabajo y devolver todos los elementos al driver (es decir, a tu notebook/Colab).
# Imprime una lista de las transacciones filtradas.

# Resultado esperado:
# Transacciones > 1000: [
# ('Ana', 1200.0, 'Supermercado'),
# ('Lucia', 2200.0, 'Tecnologia'),
# ('Juan', 3100.0, 'Tecnologia'),
# ('Carlos', 1800.0, 'Supermercado'),
# ('Ana', 2600.0, 'Tecnologia')
# ]

# ============================================
# 5) Gasto total por usuario (Pair RDD + reduceByKey)
#    (usuario -> suma de montos)
# ============================================
# En este punto, rdd_tx contiene tuplas de 3 elementos:
# (usuario, monto, categoria)

# Ejemplo:
# ("Ana", 1200.0, "Supermercado")
# ("Juan", 800.0, "Restaurante")
# ("Lucia", 2200.0, "Tecnologia")

rdd_usuario_monto = rdd_tx.map(lambda t: (t[0], t[1]))    # (usuario, monto)
# .map(...) transforma cada elemento del RDD.
# lambda t: (t[0], t[1]) significa:
# t[0] → usuario
# t[1] → monto
# El resultado es un RDD de pares (clave, valor):
# clave = usuario
# valor = monto
# Ejemplo:
# ("Ana", 1200.0)
# ("Juan", 800.0)
# ("Lucia", 2200.0)
# ("Ana", 400.0)

gasto_por_usuario = rdd_usuario_monto.reduceByKey(lambda a, b: a + b)
# reduceByKey agrupa por clave (usuario) y aplica una función de reducción sobre los valores.
# En este caso: lambda a, b: a + b → suma los montos de un mismo usuario.
# Ejemplo paso a paso:
# Para "Ana": 1200.0 + 400.0 + 2600.0 = 4200.0
# Para "Juan": 800.0 + 3100.0 = 3900.0
# Para "Lucia": 2200.0 + 950.0 = 3150.0
# Para "Carlos": 1800.0 + 700.0 = 2500.0

print("Gasto total por usuario:", gasto_por_usuario.collect())
# collect() ejecuta el plan y devuelve todos los pares (usuario, gasto total) como una lista en Python.
# Resultado esperado:
# Gasto total por usuario: [
# ('Ana', 4200.0),
# ('Juan', 3900.0),
# ('Lucia', 3150.0),
# ('Carlos', 2500.0)
# ]


Transacciones > 1000: [('Ana', 1200.0, 'Supermercado'), ('Lucia', 2200.0, 'Tecnologia'), ('Juan', 3100.0, 'Tecnologia'), ('Carlos', 1800.0, 'Supermercado'), ('Ana', 2600.0, 'Tecnologia')]
Gasto total por usuario: [('Lucia', 3150.0), ('Ana', 4200.0), ('Juan', 3900.0), ('Carlos', 2500.0)]


se filtraron todas las transacciones cuyo monto es mayor a 1000.
Cada tupla sigue la estructura (usuario, monto, categoria).

Ana gastó 1200.0 en Supermercado.

Lucia gastó 2200.0 en Tecnología.

Juan gastó 3100.0 en Tecnología.

Carlos gastó 1800.0 en Supermercado.

Ana gastó 2600.0 en Tecnología.

En total, 5 transacciones cumplen la condición (monto > 1000).

Gasto total por usuario
Gasto total por usuario: [
 ('Lucia', 3150.0),
 ('Ana', 4200.0),
 ('Juan', 3900.0),
 ('Carlos', 2500.0)
]


Significa: se agruparon todas las transacciones por usuario y se sumaron sus montos (reduceByKey).

Lucia → 2200.0 + 950.0 = 3150.0

Ana → 1200.0 + 400.0 + 2600.0 = 4200.0

Juan → 800.0 + 3100.0 = 3900.0

Carlos → 1800.0 + 700.0 = 2500.0

Este resultado muestra el total gastado por cada persona en todas sus transacciones.

En resumen:

Transacciones > 1000 → lista de compras individuales grandes.

Gasto total por usuario → cuánto gastó cada persona en total (sumando todas sus compras).

In [ ]:
# ============================================
# 6) Top 3 usuarios por gasto (sortBy descendente)
# ============================================
# En este punto ya tienes el RDD gasto_por_usuario, que contiene el gasto total de cada usuario, en forma de pares clave–valor:
# [
# ("Ana", 4200.0),
# ("Juan", 3900.0),
# ("Lucia", 3150.0),
#("Carlos", 2500.0)
# ]

top3 = gasto_por_usuario.sortBy(lambda kv: kv[1], ascending=False).take(3)
# sortBy → es una transformación que ordena los elementos de un RDD según una clave.
# lambda kv: kv[1] significa:
# kv es cada par (usuario, gasto).
# kv[1] es el gasto total (la segunda posición de la tupla).
# ascending=False → indica que el orden sea descendente (del mayor al menor gasto).
# Resultado intermedio (ordenado):
#[
# ("Ana", 4200.0),
# ("Juan", 3900.0),
# ("Lucia", 3150.0),
# ("Carlos", 2500.0)
#]

# (Si fuera ascending=True, saldría al revés).

# take(n) es una acción que devuelve los primeros n elementos.

# Como ya está ordenado, esos primeros 3 serán los usuarios con más gasto.

# Resultado esperado:
# Top 3 usuarios: [
#  ("Ana", 4200.0),
# ("Juan", 3900.0),
# ("Lucia", 3150.0)
# ]

print("Top 3 usuarios:", top3)

Top 3 usuarios: [('Ana', 4200.0), ('Juan', 3900.0), ('Lucia', 3150.0)]


Ana → 4200.0 → Ana es quien más gastó en total (sumando todas sus compras).

Juan → 3900.0 → Juan es el segundo con más gasto.

Lucia → 3150.0 → Lucia ocupa el tercer lugar.

El cuarto usuario, Carlos (con 2500.0), quedó fuera porque solo pediste el Top 3.

In [ ]:
# ============================================
# 7) Estadísticas básicas del gasto (acciones)
# ============================================

# Extraer solo los montos
montos = rdd_tx.map(lambda t: t[1])  # RDD[float]
# rdd_tx tiene tuplas de la forma (usuario, monto, categoria).
# t[1] es el campo monto.
# Con map se transforma el RDD en una secuencia solo de montos (floats).

# Ejemplo:
# [1200.0, 800.0, 2200.0, 400.0, 3100.0, 950.0, 1800.0, 700.0, 2600.0]
total = montos.sum()
# Devuelve la suma total de todos los montos.
promedio = montos.mean()
# Calcula el promedio (media aritmética).
desv = montos.stdev()
# Calcula la desviación estándar muestral, que mide la dispersión de los datos.
conteo = montos.count()
# Cuenta el número total de transacciones.

print(f"Estadísticas -> n={conteo}, total={total:.2f}, promedio={promedio:.2f}, stdev={desv:.2f}")
# Usa un f-string de Python para mostrar los valores formateados con 2 decimales (:.2f).
# Ejemplo de salida:
# Estadísticas -> n=9, total=14250.00, promedio=1583.33, stdev=905.48

# Qué practican aquí: creación de RDD desde archivo, eliminación de encabezado, map/filter, Pair RDD + reduceByKey, sortBy, y
# acciones (collect, take, sum, mean, stdev)

Estadísticas -> n=9, total=13750.00, promedio=1527.78, stdev=886.66


n=9
Hay 9 transacciones en total (es el número de registros después de quitar el encabezado del CSV).

total=13750.00
Es la suma total de todos los montos gastados.
Ejemplo: 1200 + 800 + 2200 + 400 + 3100 + 950 + 1800 + 700 + 2600 = 13750.0

promedio=1527.78

stdev=886.66
Es la desviación estándar muestral de los montos.

Mide qué tanto varían los gastos respecto al promedio.

Un valor alto (886.66) indica que los montos son bastante diversos: hay transacciones pequeñas (400, 700, 800) y grandes (2200, 2600, 3100).

En resumen:
La salida significa que hubo 9 transacciones que sumaron 13,750.00 en total; en promedio, cada transacción fue de 1,527.78 unidades monetarias, con una variabilidad de aproximadamente 886.66.

**Explicación Actividad 2**
Este ejemplo es una simulación de un archivo de logs de un servidor web, escrito en un formato muy parecido al Common Log Format (CLF), que es el estándar en servidores como Apache o Nginx para registrar cada acceso que reciben. El Common Log Format (CLF) es un formato de texto estandarizado, desarrollado por el NCSA (National Center for Supercomputing Applications), que los servidores web usan para registrar información básica sobre las solicitudes de los usuarios en un archivo de texto. Al ser un formato fijo y estandarizado, permite que programas de análisis de registros como Webalizer o AWStats puedan procesar fácilmente los datos de los archivos de registro. Cada línea en un archivo CLF contiene detalles como la dirección IP del host remoto, la marca de tiempo, la solicitud HTTP y el código de estado, entre otros.
¿Para qué sirve o es útil?
Análisis de logs:
El formato CLF facilita la lectura y el análisis de los archivos de registro por parte de software especializado como Webalizer, Analog y Awstats, permitiendo comprender patrones de tráfico y comportamiento del usuario.
Solución de problemas:
Los administradores de sistemas pueden utilizar los logs CLF para diagnosticar fallos y errores en el servidor, ya que registran información detallada sobre las solicitudes recibidas y los errores generados.
Monitoreo de seguridad:
Los datos registrados pueden ser cruciales para identificar intentos de acceso no autorizados o actividades maliciosas, ayudando a fortalecer la seguridad del sistema.
Auditorías:
Permite realizar auditorías detalladas de la actividad en el servidor y el cumplimiento de normativas, ya que cada entrada de registro proporciona datos contextuales sobre eventos específicos.
Análisis de rendimiento:
Analizando los logs con formato CLF, se puede identificar cuellos de botella, optimizar la configuración del servidor y mejorar la experiencia general del usuario.

En la práctica, cada vez que un cliente (navegador, app, etc.) hace una petición a un servidor web, se genera un registro en el log con información como:

la dirección IP del cliente,

la fecha y hora de la petición,

el método HTTP (GET, POST, etc.),

la ruta/endpoint solicitada (ejemplo /home, /checkout),

el código de estado HTTP devuelto (ejemplo 200 = OK, 404 = Not Found, 500 = Internal Server Error),

y el tamaño de la respuesta en bytes.

Ejemplo de línea en este ejercicio
10.0.0.2 - - [01/Mar/2025:10:05:05] "GET /checkout HTTP/1.1" 500 102


10.0.0.2 → IP del cliente que hizo la petición.

- - → campos de usuario/autenticación (sin usar en este ejemplo).

[01/Mar/2025:10:05:05] → fecha y hora exacta del acceso.

"GET /checkout HTTP/1.1" → método, ruta solicitada y protocolo.

500 → código HTTP → error interno del servidor.

102 → número de bytes devueltos.

¿Qué se hace con estos datos en Spark?

El objetivo es practicar procesamiento distribuido con RDDs en Spark aplicando transformaciones y acciones.

Parsing → convertir cada línea en una tupla más estructurada, por ejemplo:

("/checkout", 500)


Esto simplifica el análisis.

Filter → seleccionar solo ciertos registros. Ejemplo:

todos los errores (400 <= código < 600).

Conteo por clave → agrupar y contar:

cuántos accesos tuvo cada ruta (/home, /api/items, /checkout).

cuántos accesos hubo de cada código HTTP (200, 404, 500).

Métricas → calcular indicadores:

tasa de error = errores ÷ total de peticiones.

endpoints con más errores.

¿Qué se está practicando?

Lectura de un archivo en Spark (sc.textFile).

Transformaciones: map, filter, reduceByKey, sortBy.

Acciones: collect, count, take.

Concepto de lazy evaluation (nada se ejecuta hasta llamar una acción).

Cómo pasar de texto plano → a datos útiles para análisis.

En resumen:
Este ejemplo no es un log real, sino un log sintético (inventado) en formato similar al Common Log Format. La idea es que practiques con Spark cómo:

limpiar datos,

extraer información relevante (ruta y código),

contar accesos,

y calcular métricas como la tasa de error o los endpoints con más problemas.

**Ejemplos**
Ejemplos reales donde se usa CLF
1. Sitios web corporativos o institucionales

Cada vez que un usuario entra a la web de un banco, universidad o tienda online, el servidor (Apache o Nginx) genera una línea de log en CLF.

Ejemplo:

Un cliente consulta su saldo bancario → el servidor registra la IP, el recurso consultado, el código de respuesta (200 si todo bien, 401 si no tiene acceso).

2. Tiendas online (e-commerce)

Amazon, Mercado Libre o cualquier tienda que corra sobre Apache/Nginx genera logs CLF.

Se usan para:

Saber qué productos se visitan más.

Identificar usuarios que agregan productos al carrito pero no compran.

Detectar errores (ej. páginas 404 de productos eliminados).

3. Seguridad informática

Logs CLF sirven para detectar:

Intentos de ataque (muchos accesos fallidos con código 401 o 403).

Bots que tratan de entrar en /admin o /login.

Ataques DDoS por la cantidad de peticiones por segundo desde una misma IP.

Ejemplo real:

Un servidor de un ministerio detecta que la IP 123.45.67.89 intenta 500 accesos en 1 minuto → alerta de ataque.

4. Analítica de tráfico web

Empresas de marketing digital usan CLF para medir:

Horas de más tráfico en un sitio.

Páginas más consultadas.

Qué navegadores usan los clientes.

Ejemplo:

En un periódico digital, se detecta que la noticia “Elecciones 2025” tuvo 100 000 visitas en un día.

5. Regulación y auditoría

En sectores como salud, banca o gobierno, los logs CLF son obligatorios para:

Auditar accesos a información sensible.

Ver quién accedió a qué documento, cuándo y desde dónde.

Ejemplo:

En un hospital, los accesos al sistema de historias clínicas quedan registrados para cumplir con normas de privacidad (HIPAA, GDPR).

6. Optimización de infraestructura

Ingenieros de sistemas analizan CLF para:

Ver qué recursos consumen más ancho de banda.

Balancear carga entre servidores.

Detectar contenido no utilizado.

Ejemplo:

En una empresa de streaming, se observa que el 70% del tráfico viene de videos HD → se decide poner un CDN.

En resumen:
El CLF se usa en la vida real para:

Monitorear usuarios y tráfico.

Detectar errores y ataques.

Generar estadísticas de negocio.

Cumplir con auditorías y regulaciones.

Optimizar infraestructura.

In [ ]:
# Actividad 2: Logs web sintéticos (limpieza, conteos por clave, tasa de error)
# Objetivos: parsing de texto, filter por códigos HTTP, counts por ruta/estado, y métricas.
# ============================================
# 1) Crear log sintético tipo "Common Log Format"
#    Formato simplificado: ip - - [fecha] "GET /ruta HTTP/1.1" 200 1234
# ============================================
# 1. Objetivo de la actividad
# Esta actividad simula un archivo de logs de un servidor web en formato parecido al Common Log Format (CLF). Con esos datos después practicarás en Spark:
# Parsing (extraer campos útiles de cada línea).
# Filter por códigos HTTP (200, 404, 500…).
# Conteos por ruta.
# Cálculo de métricas como la tasa de error

logs = """
10.0.0.1 - - [01/Mar/2025:10:05:03] "GET /home HTTP/1.1" 200 532
10.0.0.2 - - [01/Mar/2025:10:05:04] "GET /api/items HTTP/1.1" 200 891
10.0.0.3 - - [01/Mar/2025:10:05:04] "GET /home HTTP/1.1" 200 480
10.0.0.2 - - [01/Mar/2025:10:05:05] "GET /checkout HTTP/1.1" 500 102
10.0.0.1 - - [01/Mar/2025:10:05:05] "GET /api/items HTTP/1.1" 404 67
10.0.0.4 - - [01/Mar/2025:10:05:06] "GET /home HTTP/1.1" 200 501
10.0.0.2 - - [01/Mar/2025:10:05:06] "GET /checkout HTTP/1.1" 200 210
10.0.0.5 - - [01/Mar/2025:10:05:07] "GET /api/items HTTP/1.1" 200 920
""".strip()
# Se define un string multilínea ("""...""") que contiene registros ficticios de accesos web.

# Cada línea representa una petición HTTP, con datos simplificados:
# 10.0.0.1 → dirección IP del cliente.
# - - → (campos de usuario y autenticación, vacíos en este ejemplo).
# [01/Mar/2025:10:05:03] → fecha y hora.
# "GET /home HTTP/1.1" → método, ruta y protocolo.
# 200 → código de estado HTTP (200=OK, 404=Not Found, 500=Internal Server Error).
# 532 → número de bytes enviados en la respuesta.

# Ejemplo:
# 10.0.0.2 - - [01/Mar/2025:10:05:05] "GET /checkout HTTP/1.1" 500 102

# Esto significa: un usuario desde la IP 10.0.0.2 pidió la ruta /checkout y el servidor devolvió un error 500 (fallo interno).
# .strip() → elimina saltos de línea al inicio y final de la cadena.

with open("/content/access.log", "w", encoding="utf-8") as f:
    f.write(logs + "\n")
# Se abre (o crea) el archivo /content/access.log en modo escritura ("w").
# encoding="utf-8" asegura compatibilidad con caracteres especiales.
# f.write(logs + "\n") → escribe el contenido de la variable logs en el archivo, más un salto de línea al final.

# 4. ¿Por qué hacerlo así?

# En Colab o en entornos de prueba:
# Muchas veces no tienes un archivo real de logs.
# Entonces lo generas "al vuelo" en el código para practicar.
# Después podrás cargarlo en Spark con:
# rdd_logs = sc.textFile("/content/access.log")

In [ ]:
# # ============================================
# 2) Cargar como RDD
# ============================================
rdd_logs = sc.textFile("/content/access.log")
# Esa línea es clave porque conecta el archivo que creaste con Spark.
# sc es el SparkContext, el objeto principal para trabajar con RDDs en Spark.
# Es el punto de entrada que permite a tu programa comunicarse con el clúster o, en este caso, con Spark en modo local (Colab).

# textFile(...) es un método de SparkContext.
# Sirve para leer un archivo de texto línea por línea.
# Cada línea del archivo se convierte en un elemento del RDD (es decir, cada registro de log será un string dentro de rdd_logs).

# Ejemplo:
# Si el archivo /content/access.log tiene:
# 10.0.0.1 - - [01/Mar/2025:10:05:03] "GET /home HTTP/1.1" 200 532
# 10.0.0.2 - - [01/Mar/2025:10:05:04] "GET /api/items HTTP/1.1" 200 891

# El RDD (rdd_logs) tendrá:
# [
# "10.0.0.1 - - [01/Mar/2025:10:05:03] \"GET /home HTTP/1.1\" 200 532",
# "10.0.0.2 - - [01/Mar/2025:10:05:04] \"GET /api/items HTTP/1.1\" 200 891"
# ]

# 3. Tipo de objeto
# rdd_logs es un RDD de strings → cada string corresponde a una línea del archivo.

# Más adelante se hará un parsing (con .map(parse_log)) para convertir esas líneas en tuplas más estructuradas como (ruta, códigoHTTP).

# 4. Ejecución diferida (lazy evaluation)

# Aunque escribas sc.textFile(...), Spark no lee el archivo inmediatamente.
# Solo crea un "plan de lectura".
# La lectura real se hace cuando aplicas una acción (ejemplo: rdd_logs.collect() o rdd_logs.take(3))

In [ ]:
# ============================================
# 3) Parseo: extraer (ruta, codigoHTTP)
#    Nota: ejemplo simple con splits; en producción usar regex robusto.
# Esta parte es donde el código convierte cada línea del log (que hasta ahora era un string largo) en algo más estructurado que Spark
# pueda procesar fácilmente.
# ============================================
# Definición de la función parse_log
# Separar por comillas dobles (")
# partes = line.split('"')
# Ejemplo de línea original:
# 10.0.0.1 - - [01/Mar/2025:10:05:03] "GET /home HTTP/1.1" 200 532
# Al dividir por ", obtienes algo así:
# [
#  "10.0.0.1 - - [01/Mar/2025:10:05:03] ",
#  "GET /home HTTP/1.1",
#  " 200 532"
# ]

def parse_log(line):
    # ... "GET /ruta HTTP/1.1" CODIGO ...
    try:
        partes = line.split('"')
        request = partes[1]                     # GET /ruta HTTP/1.1
 # De ahí puedes extraer la ruta:
# ruta = request.split(" ")[1]   # "/home"
        ruta = request.split(" ")[1]            # /ruta
        resto = partes[2].strip().split(" ")    # CODIGO BYTES
        codigo = int(resto[0]) # resto[0] = código HTTP (200, 404, 500, etc.).
        return (ruta, codigo) # Devolver la tupla
# Ejemplo:
# ("/home", 200)
# ("/checkout", 500)
    except Exception:
        return ("/_parse_error", 0)
# Si la línea no se puede procesar, se devuelve un valor por defecto para identificar errores.

rdd_parsed = rdd_logs.map(parse_log)
# .map() aplica parse_log a cada línea del RDD.
# El resultado es un nuevo RDD de tuplas (ruta, codigoHTTP).
# Ejemplo de salida:
# [
# ("/home", 200),
# ("/api/items", 200),
# ("/home", 200),
# ("/checkout", 500),
# ("/api/items", 404)
# ]

print("Parsed sample:", rdd_parsed.take(3))
# .take(3) ejecuta el plan y devuelve las 3 primeras tuplas.
# Ejemplo de salida:
# Parsed sample: [('/home', 200), ('/api/items', 200), ('/home', 200)]

# En resumen:
# Antes: cada registro era un string largo de log.
# Después: se convierte en una tupla (ruta, códigoHTTP), mucho más fácil de agrupar, filtrar o contar

Parsed sample: [('/home', 200), ('/api/items', 200), ('/home', 200)]


Cada elemento es una tupla con dos valores:

(ruta, codigoHTTP)


('/home', 200) → hubo un acceso a la ruta /home con código HTTP 200 (OK).

('/api/items', 200) → acceso a la ruta /api/items con respuesta 200 (OK).

('/home', 200) → otro acceso a /home, también exitoso.

El resultado de take(3) solo muestra los tres primeros registros parseados del log.

En otras palabras

Esto significa que Spark ya convirtió las líneas de texto crudo de los logs en datos estructurados:

qué endpoint fue llamado (/home, /api/items),

y qué resultado devolvió el servidor (códigos 200, 404, 500, etc.).

In [ ]:
# ============================================
# 4) Conteo de hits por ruta (Pair RDD + reduceByKey)
# Aquí se está midiendo cuántas veces fue visitada cada ruta en los logs
# ============================================
hits_por_ruta = rdd_parsed.map(lambda rc: (rc[0], 1)).reduceByKey(lambda a, b: a + b)
# En este punto, rdd_parsed contiene tuplas como:
# ("/home", 200)
# ("/api/items", 200)
# ("/home", 200)
# ("/checkout", 500)
# ("/api/items", 404)
# ("/home", 200)
# ("/checkout", 200)
# ("/api/items", 200)
# Cada elemento es (ruta, codigoHTTP).

# map(...) transforma cada tupla (ruta, codigo) en (ruta, 1).
# La idea es: cada vez que aparece una ruta, le damos un 1 para luego sumarlos.
# Ejemplo de salida intermedia:
# ("/home", 1)
# ("/api/items", 1)
# ("/home", 1)
# ("/checkout", 1)
# ("/api/items", 1)
# ...

# Agrupar y sumar con reduceByKey
# reduceByKey agrupa todos los valores con la misma clave (aquí, la ruta).
# Luego aplica la función de reducción lambda a, b: a + b, que suma los unos.

# Ejemplo del resultado:
# /home → 1 + 1 + 1 = 3
# /api/items → 1 + 1 + 1 = 3
# /checkout → 1 + 1 = 2
# Resultado final:
# [
# ("/home", 3),
# ("/api/items", 3),
# ("/checkout", 2)
# ]

print("Hits por ruta:", hits_por_ruta.collect())
# .collect() trae todos los resultados del RDD al programa en Python.
# El print muestra cuántos accesos tuvo cada ruta en el log.

# Ejemplo de salida:
# Hits por ruta: [('/checkout', 2), ('/api/items', 3), ('/home', 3)]
# En resumen:
# Se transforma cada línea en (ruta, 1).
# Se suman los unos por cada ruta usando reduceByKey.
# Se obtiene el número de accesos por ruta (hits).

Hits por ruta: [('/home', 3), ('/api/items', 3), ('/checkout', 2)]


Cada tupla tiene la forma:

(ruta, cantidad_de_hits)


('/home', 3) → la ruta /home fue visitada 3 veces.

('/api/items', 3) → la ruta /api/items fue visitada 3 veces.

('/checkout', 2) → la ruta /checkout fue visitada 2 veces.

De dónde viene

En el log de ejemplo, tenías 8 accesos en total:

/home apareció 3 veces.

/api/items apareció 3 veces.

/checkout apareció 2 veces.

El conteo refleja exactamente esa distribución.

En resumen:
La salida significa que Spark ya procesó los registros y te muestra cuántas peticiones recibió cada endpoint.

In [ ]:
# ============================================
# 5) Tasa de error (4xx y 5xx)
# calcula una métrica de calidad del servicio: la tasa de error en los accesos web.
# ============================================
total_hits = rdd_parsed.count()
# rdd_parsed tiene tuplas (ruta, codigoHTTP).
# .count() es una acción que devuelve cuántos elementos hay en el RDD.
# En este caso, cuántas peticiones (hits) se registraron en total.
# Ejemplo: si hay 8 líneas en el log → total_hits = 8.

# Filtrar errores HTTP
errores = rdd_parsed.filter(lambda rc: 400 <= rc[1] < 600).count()
# filter(...) mantiene solo las tuplas cuyo código HTTP (rc[1]) está entre 400 y 599.
# 4xx → errores del cliente (ej. 404 Not Found).
# 5xx → errores del servidor (ej. 500 Internal Server Error).
# .count() da el número de accesos que resultaron en error.
# Ejemplo: si hubo un 404 y un 500 → errores = 2.

# Calcular la tasa de error
tasa_error = errores / total_hits if total_hits else 0.0
# Se divide el número de errores entre el total de hits.
# El condicional if total_hits else 0.0 evita una división por cero en caso de que no haya registros.
# El resultado es un valor decimal entre 0 y 1.

# Ejemplo:
# errores = 2, total_hits = 8 → tasa_error = 2/8 = 0.25.

print(f"Total hits={total_hits}, errores={errores}, tasa_error={tasa_error:.2%}")
# Usa un f-string para formatear la salida.
# :.2% convierte el decimal en porcentaje con 2 decimales.

# Ejemplo de salida:
# Total hits=8, errores=2, tasa_error=25.00%

Total hits=8, errores=2, tasa_error=25.00%


Interpretación de cada valor

Total hits=8
→ Hubo 8 peticiones en total registradas en el log.

errores=2
→ De esas 8 peticiones, 2 devolvieron códigos de error HTTP:

4xx (errores de cliente, como 404).

5xx (errores de servidor, como 500).

tasa_error=25.00%
→ La proporción de errores sobre el total es: 2/8

Esto significa que una de cada cuatro peticiones falló.

En otras palabras

El log refleja que:

El 75% de las peticiones fueron exitosas (código 200).

El 25% restante fueron fallidas (códigos 404 o 500).

In [ ]:
# ============================================
# 6) Top por estado HTTP (cuenta por código)
# Aquí lo que se está haciendo es contar cuántas veces aparece cada código HTTP (200, 404, 500, etc.) en los logs:
# ============================================
# En este punto, rdd_parsed contiene tuplas como:
("/home", 200)
# ("/api/items", 200)
# ("/home", 200)
# ("/checkout", 500)
# ("/api/items", 404)
# ("/home", 200)
# ("/checkout", 200)
# ("/api/items", 200)
# Cada registro tiene: (ruta, codigoHTTP).
por_codigo = rdd_parsed.map(lambda rc: (rc[1], 1)).reduceByKey(lambda a, b: a + b)
# rc es la tupla (ruta, codigo).
# rc[1] es el código HTTP.
# Se transforma cada elemento en (codigoHTTP, 1), para poder contarlos.
# Ejemplo del resultado intermedio:
# (200, 1)
# (200, 1)
# (200, 1)
# (500, 1)
# (404, 1)
# (200, 1)
# (200, 1)
# (200, 1)

# reduceByKey agrupa por la clave (el código HTTP) y aplica la función de reducción a + b, que suma los valores.
# Básicamente, está sumando los unos de cada código.
# Resultado final:
# [
# (200, 6),   # 6 respuestas exitosas
# (500, 1),   # 1 error de servidor
# (404, 1)    # 1 error de cliente
# ]

print("Conteo por código:", por_codigo.collect())
# .collect() devuelve todo el RDD como una lista en Python.
# El print muestra el conteo por código HTTP.
# Ejemplo de salida:
# Conteo por código: [(500, 1), (404, 1), (200, 6)]

Conteo por código: [(200, 6), (500, 1), (404, 1)]


Es una lista de tuplas (códigoHTTP, cantidad):
(200, 6) → hubo 6 accesos exitosos con código 200 OK.
(500, 1) → hubo 1 error de servidor con código 500 Internal Server Error.
(404, 1) → hubo 1 error de cliente con código 404 Not Found.
Relación con los logs de ejemplo

Si recuerdamos, el log tenías 8 líneas:
6 peticiones con 200 → éxito.
1 petición con 500 → error interno del servidor.
1 petición con 404 → recurso no encontrado.
La salida refleja exactamente esa distribución.

En otras palabras
La salida te está diciendo:
“De todas las 8 peticiones registradas, 6 fueron exitosas (200), 1 falló por error de servidor (500) y 1 falló porque el recurso no existía (404)”.

200, 6) → Hubo 6 peticiones exitosas (código HTTP 200 = OK).

(500, 1) → Hubo 1 error de servidor (código HTTP 500 = Internal Server Error).

(404, 1) → Hubo 1 error de cliente (código HTTP 404 = Not Found).

Relación con el log

En el archivo access.log tenías 8 accesos:

6 devolvieron código 200.

1 devolvió código 500.

1 devolvió código 404.

Por eso el conteo coincide exactamente con esa distribución.

En resumen:
La salida significa que, de todas las peticiones, la gran mayoría (6/8 = 75%) fueron exitosas, mientras que solo un 25% fueron errores (1 de tipo 500 y 1 de tipo 404).

In [ ]:
# ============================================
# 7) Endpoints con mayor error (filtrar 4xx/5xx, contar por ruta, ordenar)
# última parte del código responde a una pregunta muy común en análisis de logs: ¿qué endpoints (rutas) generan más errores?.
# Hasta aquí, tienes rdd_parsed con tuplas:
# (ruta, codigoHTTP)
# Ejemplo:
# ("/home", 200)
# ("/api/items", 200)
# ("/checkout", 500)
# ("/api/items", 404)
# ("/checkout", 200)
# ("/api/items", 200)
# ============================================
errores_por_ruta = (rdd_parsed
                    .filter(lambda rc: 400 <= rc[1] < 600)
                    .map(lambda rc: (rc[0], 1))
                    .reduceByKey(lambda a, b: a + b)
                    .sortBy(lambda kv: kv[1], ascending=False))
# Filtrar solo errores (4xx y 5xx)
# rc[1] es el código HTTP.
# Se quedan solo los que están entre 400 y 599.
# Resultado intermedio:
# ("/checkout", 500)
# ("/api/items", 404)

# map(lambda rc: (rc[0], 1))
# Transforma cada error en (ruta, 1).
# Resultado:
# ("/checkout", 1)
# ("/api/items", 1)

# reduceByKey(lambda a, b: a + b)
# Agrupa por clave (la ruta) y suma los 1.
# Resultado:
# ("/checkout", 1)
# ("/api/items", 1)
# (en este dataset, cada una tuvo 1 error).

# .sortBy(lambda kv: kv[1], ascending=False)
# Ordena las tuplas (ruta, cantidadErrores) por el valor (kv[1]).
# ascending=False → de mayor a menor.

# Resultado final:
# [("/checkout", 1), ("/api/items", 1)]

print("Rutas con más errores:", errores_por_ruta.collect())
# .collect() fuerza a Spark a ejecutar todas las transformaciones.
# Muestra las rutas ordenadas por cantidad de errores.

# Salida esperada:
# Rutas con más errores: [('/checkout', 1), ('/api/items', 1)]

# Este código encuentra qué endpoints producen más errores, los cuenta y los ordena para identificar los más problemáticos.

# Qué practican: parsing de texto a tuplas, Pair RDD para conteos, filter, reduceByKey, count, sortBy y cálculo de
# métricas (tasa de error). Recalca el concepto de lazy evaluation: nada corre hasta count()/collect()/take()

Rutas con más errores: [('/api/items', 1), ('/checkout', 1)]


Cada tupla tiene la forma:

(ruta, cantidad_de_errores)


('/api/items', 1) → La ruta /api/items tuvo 1 error (en este caso, un 404 Not Found).

('/checkout', 1) → La ruta /checkout tuvo 1 error (un 500 Internal Server Error).

Relación con los datos

En tu log de ejemplo:

/api/items aparece varias veces, pero una de ellas devolvió código 404 → por eso tiene 1 error.

/checkout aparece dos veces, una con código 200 OK y otra con 500 Internal Server Error → por eso tiene 1 error.

/home siempre devolvió 200 OK, así que no aparece en esta lista.

En resumen:
La salida significa que las rutas /api/items y /checkout son las que generaron errores en el log, cada una con un error registrado.

In [ ]:
# Actividad 3: IoT + Join entre RDDs (persistencia, aggregate, joins)
# Objetivos: cargar dos CSV, calcular promedios por dispositivo, persistir resultados, y hacer un join con metadatos.
# ============================================
# Esta parte solo construye dos archivos CSV de ejemplo en Colab para luego cargarlos con Spark y practicar agregaciones y joins.
# 1) CSV sensores y CSV metadatos
# ============================================
# Definir los datos como texto multilínea
sensores = """ts,device_id,temperatura,humedad
2025-03-10T10:00:00Z,A1,22.5,40
2025-03-10T10:05:00Z,A2,28.0,35
2025-03-10T10:10:00Z,A1,23.1,41
2025-03-10T10:15:00Z,A3,21.0,50
2025-03-10T10:20:00Z,A2,29.3,36
2025-03-10T10:25:00Z,A1,24.0,39
"""
# Crea una cadena multilínea que imita un CSV de sensores IoT.
# Columnas:
# ts: marca de tiempo (ISO-8601 con “Z”, UTC).
# device_id: id del dispositivo (A1, A2, A3).
# temperatura: valor numérico (float).
# humedad: valor numérico (entero o float).
# Observación: el encabezado está en la primera línea; luego vienen los registros.

meta = """device_id,ubicacion,modelo
A1,Planta Norte,T-100
A2,Planta Sur,T-200
A3,Planta Norte,T-150
"""
# Segundo CSV con metadatos por dispositivo.
# Columnas:
# device_id: clave para enlazar.
# ubicacion: dónde está instalado.
# modelo: referencia del equipo.
# ¿Para qué? Luego haremos un join por device_id entre mediciones y metadatos, de modo que a cada promedio por dispositivo se
#le pueda anexar su ubicación/modelo.

# Guardar los textos como archivos en Colab
with open("/content/sensores.csv", "w", encoding="utf-8") as f:
    f.write(sensores)
with open("/content/meta.csv", "w", encoding="utf-8") as f:
    f.write(meta)
# Abre/crea los archivos en /content/ (ruta típica de Google Colab).
# Modo "w": escritura (sobrescribe si existe).
# encoding="utf-8": evita problemas con acentos y caracteres especiales.
# Escribe el contenido de las variables sensores y meta en sus archivos.

In [ ]:
# ============================================
# 2) Cargar RDDs y remover encabezado
# ============================================
# Leer los archivos como RDDs
rdd_sens = sc.textFile("/content/sensores.csv")
rdd_meta = sc.textFile("/content/meta.csv")
# sc.textFile(...) lee un archivo de texto línea por línea.
# Cada línea del CSV se convierte en un string dentro de un RDD.
# Ejemplo (rdd_sens.take(2)):
# [
# "ts,device_id,temperatura,humedad",
# "2025-03-10T10:00:00Z,A1,22.5,40"
# ]

# Ejemplo (rdd_meta.take(2)):
# [
# "device_id,ubicacion,modelo",
# "A1,Planta Norte,T-100"
# ]

# Identificar los encabezados
head_s = rdd_sens.first()
head_m = rdd_meta.first()
# .first() devuelve el primer elemento del RDD.
# En este caso, cada primer elemento corresponde al encabezado de columnas.
# Valores guardados:
# head_s = "ts,device_id,temperatura,humedad"
# head_m = "device_id,ubicacion,modelo"

# Filtrar para remover encabezados
rdd_sens = rdd_sens.filter(lambda l: l != head_s)
rdd_meta = rdd_meta.filter(lambda l: l != head_m)
# .filter(...) genera un nuevo RDD que solo mantiene las líneas distintas del encabezado.
# lambda l: l != head_s → elimina la línea que es igual a la cabecera de sensores.
# lambda l: l != head_m → elimina la línea que es igual a la cabecera de metadatos.
# Ahora:
# rdd_sens tiene solo las mediciones (sin la fila ts,device_id,...).
# rdd_meta tiene solo los metadatos (sin la fila device_id,ubicacion,...).

# En resumen:
# Se cargan los CSV como RDDs de texto.
# Se obtiene la primera línea (encabezado).
# Se elimina esa línea para trabajar solo con los datos útiles.

In [ ]:
# ============================================
# 3) Parseo
#    sensores: (device_id, (temp, 1))
#    meta:     (device_id, (ubicacion, modelo))
# ============================================
# Función parse_sensor
def parse_sensor(line):
    # ts,device_id,temperatura,humedad
    ts, dev, temp, hum = line.split(",")
    return (dev, (float(temp), 1))
# Cada línea del CSV de sensores es un string como:
# 2025-03-10T10:00:00Z,A1,22.5,40
# line.split(",") → divide la línea en partes:
# ["2025-03-10T10:00:00Z", "A1", "22.5", "40"]
# Asignación:
# ts → "2025-03-10T10:00:00Z" (no se usa aquí).
# dev → "A1" (id del dispositivo).
# temp → "22.5" (temperatura, string).
# hum → "40" (humedad, string, tampoco se usa).
# Devuelve una tupla clave-valor:
# Clave: dev (ej. "A1").
# Valor: (float(temp), 1) → la temperatura convertida a número y un contador 1 (se usará para calcular promedios).
# Ejemplo de salida:
# ("A1", (22.5, 1))

# Función parse_meta
def parse_meta(line):
    dev, ubic, modelo = line.split(",")
    return (dev, (ubic, modelo))
# Cada línea del CSV de metadatos es algo como:
# A1,Planta Norte,T-100
# line.split(",") →
# ["A1", "Planta Norte", "T-100"]
# Asignación:
# dev → "A1" (id del dispositivo).
# ubic → "Planta Norte" (ubicación).
# modelo → "T-100" (modelo del sensor).
# Devuelve tupla clave-valor:
# Clave: dev.
# Valor: (ubic, modelo).
# Ejemplo de salida:
# ("A1", ("Planta Norte", "T-100"))

# Crear los Pair RDDs
pair_sens = rdd_sens.map(parse_sensor)
pair_meta = rdd_meta.map(parse_meta)
# .map(parse_sensor) aplica la función de parsing a cada línea de sensores.
# .map(parse_meta) aplica la función de parsing a cada línea de metadatos.
# Ejemplo de pair_sens.take(2):
# [("A1", (22.5, 1)), ("A2", (28.0, 1))]
# Ejemplo de pair_meta.collect():
#
# [("A1", ("Planta Norte", "T-100")),
# ("A2", ("Planta Sur", "T-200")),
# ("A3", ("Planta Norte", "T-150"))]

# En resumen:
# parse_sensor prepara datos para cálculos estadísticos (temperatura promedio por dispositivo).
# parse_meta prepara datos descriptivos (ubicación y modelo).
# Ambos generan Pair RDDs con device_id como clave, lo que permitirá más adelante hacer joins y agregaciones.

In [ ]:
# ============================================
# 4) Promedio de temperatura por dispositivo
#    aggregateByKey para sumar temperaturas y contarlas
# ============================================
# Esta parte es donde realmente se hace la estadística: calcular el promedio de temperatura por cada dispositivo IoT.
# Hasta aquí, pair_sens contiene tuplas clave–valor:
# ("A1", (22.5, 1))
# ("A2", (28.0, 1))
# ("A1", (23.1, 1))
# ("A3", (21.0, 1))
# ("A2", (29.3, 1))
# ("A1", (24.0, 1))
# Clave → device_id (ej. "A1", "A2", "A3").
# Valor → (temperatura, 1) (la lectura y un contador que siempre vale 1).
# El contador sirve para poder calcular el promedio = suma de temperaturas / número de lecturas.
suma_cont = pair_sens.aggregateByKey((0.0, 0),
                                     lambda acc, v: (acc[0] + v[0], acc[1] + v[1]),
                                     lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1]))
# Función general de aggregateByKey:
# aggregateByKey(valor_inicial, func_local, func_global)
# valor_inicial: estado inicial para cada clave → (0.0, 0) (suma = 0, contador = 0).
# func_local: combina el acumulador con un valor dentro de la misma partición.
# func_global: combina acumuladores de distintas particiones.

# Desglose de las funciones:
# lambda acc, v: (acc[0] + v[0], acc[1] + v[1])
# acc = acumulador (suma, conteo)
# v = valor (temp, 1)
# Devuelve una nueva tupla con la suma y el conteo actualizados.

# Ejemplo para "A1":
# (0.0, 0) + (22.5, 1) → (22.5, 1)
# (22.5, 1) + (23.1, 1) → (45.6, 2)
# (45.6, 2) + (24.0, 1) → (69.6, 3)

# lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
# Combina acumuladores de distintas particiones (misma lógica: sumar sumas y contadores).
# Resultado suma_cont:
# [
# ("A1", (69.6, 3)),   # suma temperaturas=69.6, conteo=3
# ("A2", (57.3, 2)),   # suma=57.3, conteo=2
# ("A3", (21.0, 1))    # suma=21.0, conteo=1
# ]

promedio_temp = suma_cont.mapValues(lambda sc: sc[0] / sc[1] if sc[1] else None)
# .mapValues(...) aplica la función solo a los valores de cada par (clave, valor).
# sc = (suma, conteo).
# Devuelve suma/conteo si conteo > 0, si no → None.
# Resultado promedio_temp:
# [
# ("A1", 23.2),   # 69.6 / 3
# ("A2", 28.65),  # 57.3 / 2
# ("A3", 21.0)    # 21.0 / 1
# ]

# Persistimos porque lo usaremos varias veces (memoria) -> acelera reusos
promedio_temp.persist()
# Como este RDD promedio_temp se va a usar varias veces (por ejemplo, para joins y ordenamientos), lo guardamos en memoria.
# Esto evita recalcular todas las operaciones previas cada vez.

print("Promedio temperatura por device:", promedio_temp.collect())
# .collect() ejecuta el DAG y devuelve todos los resultados como lista en Python.
# Salida esperada:
# Promedio temperatura por device: [('A1', 23.2), ('A2', 28.65), ('A3', 21.0)]

Promedio temperatura por device: [('A1', 23.2), ('A2', 28.65), ('A3', 21.0)]


Es una lista de tuplas en Python.
Cada tupla tiene la forma:

(device_id, promedio_temperatura)

Interpretación

('A1', 23.2) → El dispositivo A1 tuvo en promedio 23.2 °C en sus mediciones.

('A2', 28.65) → El dispositivo A2 registró en promedio 28.65 °C.

('A3', 21.0) → El dispositivo A3 tuvo un promedio de 21.0 °C (solo una lectura, por eso coincide con su valor único).

Cómo se llegó a esos valores

Tomando todas las lecturas de temperatura de cada device_id y calculando:

A1 → Lecturas: 22.5, 23.1, 24.0
Promedio = (22.5 + 23.1 + 24.0) / 3 = 23.2

A2 → Lecturas: 28.0, 29.3
Promedio = (28.0 + 29.3) / 2 = 28.65

A3 → Lectura: 21.0
Promedio = 21.0

In [ ]:
# ============================================
# 5) Join con metadatos (Pair RDD join)
#    Resultado: device -> (promedio_temp, (ubicacion, modelo))
# ============================================
# Aquí viene la parte en la que unes los resultados estadísticos con la información descriptiva de cada dispositivo.
# Hasta aquí tienes dos Pair RDDs (clave–valor):

# Promedio de temperatura por dispositivo (promedio_temp)
# ("A1", 23.2)
# ("A2", 28.65)
# ("A3", 21.0)
# Metadatos (pair_meta)
# ("A1", ("Planta Norte", "T-100"))
# ("A2", ("Planta Sur", "T-200"))
# ("A3", ("Planta Norte", "T-150"))
# En ambos, la clave es el device_id.
# Por eso pueden hacerse un join directo.
joinado = promedio_temp.join(pair_meta)
# join combina dos Pair RDDs con la misma clave (device_id).
# El valor resultante es una tupla con los valores de ambos RDDs.
# Resultado joinado:
# [
# ("A1", (23.2, ("Planta Norte", "T-100"))),
#  ("A2", (28.65, ("Planta Sur", "T-200"))),
#  ("A3", (21.0, ("Planta Norte", "T-150")))
# ]

# Ordenar por temperatura promedio descendente
top_by_temp = joinado.sortBy(lambda kv: kv[1][0], ascending=False).collect()
# sortBy(...) ordena los elementos.
# kv es cada par (device_id, (promedio, (ubic, modelo))).
# kv[1][0] → accede al promedio de temperatura.
# ascending=False → orden descendente (del más alto al más bajo).
# .collect() → trae todo a Python.

# Resultado esperado:
# [
# ("A2", (28.65, ("Planta Sur", "T-200"))),
# ("A1", (23.2, ("Planta Norte", "T-100"))),
# ("A3", (21.0, ("Planta Norte", "T-150")))
# ]

print("Dispositivos por temperatura desc:", top_by_temp)
# Dispositivos por temperatura desc: [
# ('A2', (28.65, ('Planta Sur', 'T-200'))),
# ('A1', (23.2, ('Planta Norte', 'T-100'))),
# ('A3', (21.0, ('Planta Norte', 'T-150')))
# ]
# Interpretación
# El dispositivo A2 tuvo la mayor temperatura promedio (28.65 °C) y está en la Planta Sur (modelo T-200).
# El dispositivo A1 registró 23.2 °C, en la Planta Norte (modelo T-100).
# El dispositivo A3 tuvo la menor temperatura promedio (21.0 °C), también en la Planta Norte (modelo T-150).

Dispositivos por temperatura desc: [('A2', (28.65, ('Planta Sur', 'T-200'))), ('A1', (23.2, ('Planta Norte', 'T-100'))), ('A3', (21.0, ('Planta Norte', 'T-150')))]


In [ ]:
# ============================================
# 6) Reporte legible
# ============================================
# Esta parte toma el resultado ya ordenado (top_by_temp) y lo imprime en un formato humano-legible.
# Estructura de top_by_temp
# Del paso anterior, cada elemento de top_by_temp tiene la forma:
# (device_id, (promedio_temp, (ubicacion, modelo)))
# Ejemplo:
# ("A2", (28.65, ("Planta Sur", "T-200")))

# Desempaquetado (tuple unpacking) en el for
for dev, (tmean, (ubic, modelo)) in top_by_temp:
    print(f"Device {dev} ({modelo}, {ubic}) -> T° promedio = {tmean:.2f} °C")
# dev ← device_id (p.ej., "A2").
#vtmean ← promedio de temperatura (p.ej., 28.65).
# ubic ← ubicación (p.ej., "Planta Sur").

# print(f"Device {dev} ({modelo}, {ubic}) -> T° promedio = {tmean:.2f} °C")
# f"..." es un f-string de Python.
# {dev}, {modelo}, {ubic} insertan sus valores directamente.
# {tmean:.2f} formatea el número con 2 decimales.
# El símbolo °C se imprime tal cual (texto), no afecta al número.

# Salida de ejemplo:
# Device A2 (T-200, Planta Sur) -> T° promedio = 28.65 °C
# Device A1 (T-100, Planta Norte) -> T° promedio = 23.20 °C
# Device A3 (T-150, Planta Norte) -> T° promedio = 21.00 °C
# ¿Por qué así?

# Ya habías ordenado por temperatura descendente, así que se imprime de mayor a menor.
# El reporte es legible para usuarios no técnicos y útil para presentar resultados en clase.

# modelo ← modelo del sensor (p.ej., "T-200").
# Este patrón de desempaquetado te evita escribir índices como kv[1][0], haciendo el código más claro.

# Qué practican: aggregateByKey (patrón de suma/contador), persistencia de RDD para reuso, join entre Pair RDDs y ordenamientos.
# También pueden abrir el UI de Spark (http://localhost:4040) para ver jobs / stages / tasks tras ejecutar acciones

Device A2 (T-200, Planta Sur) -> T° promedio = 28.65 °C
Device A1 (T-100, Planta Norte) -> T° promedio = 23.20 °C
Device A3 (T-150, Planta Norte) -> T° promedio = 21.00 °C


Cada línea describe un dispositivo con:

ID del dispositivo (A2, A1, A3).

Modelo (ej. T-200, T-100, T-150).

Ubicación (ej. Planta Sur, Planta Norte).

Temperatura promedio registrada en el periodo de datos (calculada con aggregateByKey).

Desglose

Device A2 (T-200, Planta Sur) → en la Planta Sur, el dispositivo modelo T-200 registró un promedio de 28.65 °C, siendo el más alto.

Device A1 (T-100, Planta Norte) → en la Planta Norte, modelo T-100, promedio de 23.20 °C.

Device A3 (T-150, Planta Norte) → también en Planta Norte, modelo T-150, promedio de 21.00 °C, el más bajo.

En resumen:
La salida te está diciendo, de manera clara, qué dispositivo tuvo mayor o menor temperatura promedio, junto con su modelo y ubicación.
Sirve como informe final para identificar:

Zonas más calientes o frías.

Dispositivos con condiciones críticas.

Comparaciones rápidas entre equipos y lugares.